In [29]:
import requests
import pandas as pd
from datetime import datetime, timedelta

def fetch_snapshot(coin_ids=['bitcoin', 'ethereum', 'tether'], vs_currency='usd'):
    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        'vs_currency': vs_currency,
        # 'ids': ','.join(coin_ids),
        'price_change_percentage': '1h,24h,7d'
    }
    response = requests.get(url, params=params).json()

    records = []
    for coin in response:
        records.append({
            'coin': coin['name'],
            'symbol': coin['symbol'].upper(),
            'price': coin['current_price'],
            '1h': coin.get('price_change_percentage_1h_in_currency', 0),
            '24h': coin.get('price_change_percentage_24h_in_currency', 0),
            '7d': coin.get('price_change_percentage_7d_in_currency', 0),
            '24h_volume': coin['total_volume'],
            'mkt_cap': coin['market_cap'],
            'date': (datetime.now().date()-timedelta(days=4)).strftime('%Y-%m-%d')
        })

    df = pd.DataFrame(records)
    return df


In [26]:
datetime.now().date()-timedelta(days=4)


datetime.date(2025, 9, 21)

In [33]:
df =fetch_snapshot()
df.head()
df.coin.unique()

array(['Bitcoin', 'Ethereum', 'XRP', 'Tether', 'BNB', 'Solana', 'USDC',
       'Dogecoin', 'Lido Staked Ether', 'TRON', 'Cardano',
       'Wrapped stETH', 'Chainlink', 'Wrapped Beacon ETH', 'Ethena USDe',
       'Wrapped Bitcoin', 'Avalanche', 'Figure Heloc', 'Hyperliquid',
       'Sui', 'Stellar', 'Bitcoin Cash', 'Wrapped eETH', 'WETH', 'Hedera',
       'LEO Token', 'USDS', 'Litecoin',
       'Binance Bridged USDT (BNB Smart Chain)', 'Toncoin', 'Shiba Inu',
       'Cronos', 'Coinbase Wrapped BTC', 'Polkadot', 'WhiteBIT Coin',
       'Ethena Staked USDe', 'Mantle', 'World Liberty Financial',
       'Monero', 'USDT0', 'Uniswap', 'Dai', 'Aave', 'MemeCore', 'Ethena',
       'Pepe', 'Aster', 'NEAR Protocol', 'OKB', 'Story', 'Bitget Token',
       'Jito Staked SOL', 'Aptos', 'Bittensor', 'Ondo',
       'Ethereum Classic', 'Worldcoin', 'Binance Staked SOL', 'USD1',
       'Binance-Peg WETH', 'POL (ex-MATIC)', 'Arbitrum',
       'Internet Computer', 'Pi Network', 'sUSDS',
       'Jupiter Perp

In [32]:
print(df.shape)
df.info()

(100, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   coin        100 non-null    object 
 1   symbol      100 non-null    object 
 2   price       100 non-null    float64
 3   1h          100 non-null    float64
 4   24h         100 non-null    float64
 5   7d          99 non-null     float64
 6   24h_volume  100 non-null    float64
 7   mkt_cap     100 non-null    int64  
 8   date        100 non-null    object 
dtypes: float64(5), int64(1), object(3)
memory usage: 7.2+ KB


In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime

# List of coins (CoinGecko IDs must be lowercase and hyphenated)
coin_names = ['bitcoin', 'ethereum', 'tether', 'bitget-token', 'jito-staked-sol',
       'immutable', 'polygon-bridged-usdt-polygon', 'quant']

# Normalize to CoinGecko IDs (lowercase, hyphenated)
def normalize_name(name):
    return name.lower().replace(' ', '-').replace('(', '').replace(')', '').replace('.', '').replace('/', '-')

coin_ids = [normalize_name(name) for name in coin_names]

# Fetch 90 days of daily price data
def fetch_coin_data(coin_id, days=200):
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart"
    params = {'vs_currency': 'usd', 'days': days, 'interval': 'daily'}
    try:
        response = requests.get(url, params=params)
        data = response.json()
        df = pd.DataFrame(data['prices'], columns=['timestamp', 'price'])
        df['date'] = pd.to_datetime(df['timestamp'], unit='ms').dt.date-timedelta(days=5)
        df['coin'] = coin_id
        return df[['date', 'coin', 'price']]
    except Exception as e:
        print(f"❌ Failed for {coin_id}: {e}")
        return pd.DataFrame()

# Aggregate all coins
all_data = []
for coin_id in coin_ids:
    df = fetch_coin_data(coin_id)
    if not df.empty:
        all_data.append(df)
    time.sleep(1.2)  # Respect API rate limits

df_all = pd.concat(all_data).reset_index(drop=True)
print(f"✅ Loaded {df_all.shape[0]} rows across {len(all_data)} coins")

# Optional: Save to CSV
df_all.to_csv("crypto_timeseries_90d.csv", index=False)


❌ Failed for immutable: 'prices'
❌ Failed for polygon-bridged-usdt-polygon: 'prices'
❌ Failed for quant: 'prices'
✅ Loaded 1005 rows across 5 coins


In [ ]:
df_all.coin.unique()
print(df_all.coin.value_counts())
# df_all.to_csv("crypto_timeseries_90d.csv", index=False)




coin
bitcoin            201
ethereum           201
tether             201
bitget-token       201
jito-staked-sol    201
Name: count, dtype: int64


In [60]:
df = df_all.sort_values(['coin', 'date']).reset_index(drop=True)

# Compute percentage changes
df['pct_change_1d'] = df.groupby('coin')['price'].pct_change(periods=1)
df['pct_change_7d'] = df.groupby('coin')['price'].pct_change(periods=7)
df['pct_change_24h'] = df['pct_change_1d']  # assuming daily frequency

# Fill volume and market cap with placeholders (for now)
df['volume_24h'] = 1e9  # dummy constant or use random.normal if needed
df['mkt_cap'] = df['price'] * 1e4  # proxy based on price

# Reorder columns to match pipeline
df = df[['date', 'coin', 'price', 'pct_change_1d', 'pct_change_24h',
         'pct_change_7d', 'volume_24h', 'mkt_cap']]

#Rename to match your pipeline exactly:
df.rename(columns={
    'pct_change_1d': 'pct_change_1h',  # if needed for compatibility
}, inplace=True)



In [63]:
df.head()
df.info()

#Imputation Strategy

for col in ['pct_change_1h', 'pct_change_24h', 'pct_change_7d']:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

print("🔧 Imputed missing values:")
for col in ['pct_change_1h', 'pct_change_24h', 'pct_change_7d']:
    missing = df[col].isna().sum()
    if missing > 0:
        print(f"  {col}: {missing} filled with median {df[col].median():.4f}")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            1005 non-null   object 
 1   coin            1005 non-null   object 
 2   price           1005 non-null   float64
 3   pct_change_1h   1000 non-null   float64
 4   pct_change_24h  1000 non-null   float64
 5   pct_change_7d   970 non-null    float64
 6   volume_24h      1005 non-null   float64
 7   mkt_cap         1005 non-null   float64
dtypes: float64(6), object(2)
memory usage: 62.9+ KB
🔧 Imputed missing values:


In [64]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            1005 non-null   object 
 1   coin            1005 non-null   object 
 2   price           1005 non-null   float64
 3   pct_change_1h   1005 non-null   float64
 4   pct_change_24h  1005 non-null   float64
 5   pct_change_7d   1005 non-null   float64
 6   volume_24h      1005 non-null   float64
 7   mkt_cap         1005 non-null   float64
dtypes: float64(6), object(2)
memory usage: 62.9+ KB


In [65]:
import pickle

with open("live_crypto_data_clean.pkl", "wb") as f:
    pickle.dump(df, f)

print("✅ Clean data saved to live_crypto_data_clean.pkl")

✅ Clean data saved to live_crypto_data_clean.pkl


In [ ]:
# Load the cleaned data
with open("live_crypto_data_clean.pkl", "rb") as f:
    df_live = pickle.load(f)

# Build features
df_feat = build_features(df_live, regime_aware=True)

# Run ensemble
metrics, df_feat, primary_models, top_feats, fold_outputs = run_wfv_regime_routed_ensemble(
    df_feat,
    fallback_weights={1: 0.7, 2: 0.7, 4: 0.7},
    residual_weight=1.0,
    top_n_features=25,
    regime_aware=True,
    blend_with_lag=5,
    shap_lock=True,
    verbose=True
)

In [ ]:
from src.data_loader import crypto_data_ingestion, perform_data_preprocessing
from src.feature_engineering import add_liquidity_feature
import src.config as config
import pandas as pd
import pickle

df1 = pd.read_csv('../datasets/coin_gecko_2022-03-16.csv')
df2 = pd.read_csv('../datasets/coin_gecko_2022-03-17.csv')
df_historical = pd.concat([df1, df2])    

df_historical = perform_data_preprocessing(df_historical)
df_historical = add_liquidity_feature(df_historical)
print(df_historical.info())
with open("../notebooks/crypto_training_data.pkl", "wb") as f:
    pickle.dump(df_historical, f)

print("✅ Clean historical data saved to crypto_training_data.pkl")

Missing values before fill: 
 coin              0
symbol            0
price             0
pct_change_1h     7
pct_change_24h    7
pct_change_7d     8
volume_24h        7
mkt_cap           0
date              0
dtype: int64

 Missing values after fill: coin              0
symbol            0
price             0
pct_change_1h     0
pct_change_24h    0
pct_change_7d     0
volume_24h        0
mkt_cap           0
date              0
dtype: int64
Null dates after conversion:  0
Index(['coin', 'symbol', 'price', 'pct_change_1h', 'pct_change_24h',
       'pct_change_7d', 'volume_24h', 'mkt_cap', 'date'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   coin                   1000 non-null   object        
 1   symbol                 1000 non-null   object        
 2   price                  1000

c:\Projects\PythonFiles\MachineLearning\crypto\crypto\src\data_loader.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)
c:\Projects\PythonFiles\MachineLearning\crypto\crypto\src\data_loader.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a co

In [1]:
import pandas as pd
import pickle
import os
from src.data_loader import perform_data_preprocessing
from src.feature_engineering import add_liquidity_feature

def convert_csvs_to_pkl(csv_paths, output_path):
    # Load and concatenate CSVs
    dfs = [pd.read_csv(path) for path in csv_paths]
    df_combined = pd.concat(dfs, ignore_index=True)

    # Run preprocessing pipeline
    df_clean = perform_data_preprocessing(df_combined)
    df_clean = add_liquidity_feature(df_clean)

    # Save to .pkl
    with open(output_path, "wb") as f:
        pickle.dump(df_clean, f)

    print(f"✅ Saved preprocessed data to: {output_path}")

In [2]:
convert_csvs_to_pkl(
    csv_paths=["../datasets/coin_gecko_2022-03-17.csv"],
    output_path="../notebooks/crypto_test_data.pkl"
)

Missing values before fill: 
 coin              0
symbol            0
price             0
pct_change_1h     4
pct_change_24h    4
pct_change_7d     5
volume_24h        4
mkt_cap           0
date              0
dtype: int64

 Missing values after fill: coin              0
symbol            0
price             0
pct_change_1h     0
pct_change_24h    0
pct_change_7d     0
volume_24h        0
mkt_cap           0
date              0
dtype: int64
Null dates after conversion:  0
Index(['coin', 'symbol', 'price', 'pct_change_1h', 'pct_change_24h',
       'pct_change_7d', 'volume_24h', 'mkt_cap', 'date'],
      dtype='object')
✅ Saved preprocessed data to: ../notebooks/crypto_test_data.pkl


c:\Projects\PythonFiles\MachineLearning\crypto\crypto\src\data_loader.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)
c:\Projects\PythonFiles\MachineLearning\crypto\crypto\src\data_loader.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a co

In [16]:
import pandas as pd
import numpy as np
df_sample = pd.read_csv('../sample/crypto_sentiment_prediction_dataset.csv')
df_sample.head(), df_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2063 entries, 0 to 2062
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   timestamp                 2063 non-null   object 
 1   cryptocurrency            2063 non-null   object 
 2   current_price_usd         2063 non-null   float64
 3   price_change_24h_percent  2063 non-null   float64
 4   trading_volume_24h        2063 non-null   float64
 5   market_cap_usd            2063 non-null   float64
 6   social_sentiment_score    2063 non-null   float64
 7   news_sentiment_score      2063 non-null   float64
 8   news_impact_score         2063 non-null   float64
 9   social_mentions_count     2063 non-null   int64  
 10  fear_greed_index          2063 non-null   float64
 11  volatility_index          2063 non-null   float64
 12  rsi_technical_indicator   2063 non-null   float64
 13  prediction_confidence     2063 non-null   float64
dtypes: float

(        timestamp cryptocurrency  current_price_usd  price_change_24h_percent  \
 0  6/4/2025 20:36       Algorand             0.3427                     -5.35   
 1  6/4/2025 20:48         Cosmos            12.0420                      5.14   
 2  6/4/2025 21:28         Cosmos            11.7675                     -6.12   
 3  6/4/2025 21:57       Ethereum          2861.2829                    -11.54   
 4  6/4/2025 22:06         Solana            95.3583                      5.79   
 
    trading_volume_24h  market_cap_usd  social_sentiment_score  \
 0          1716266.10    1.762124e+09                   0.367   
 1         10520739.91    2.099180e+11                  -0.278   
 2           642191.11    1.755370e+11                  -0.255   
 3          5356227.76    4.786420e+13                  -0.531   
 4           735971.56    2.667610e+11                   0.369   
 
    news_sentiment_score  news_impact_score  social_mentions_count  \
 0                 0.374              

In [17]:
df_sample.rename(columns={
    'cryptocurrency': 'coin',
    'current_price_usd': 'price',
    'price_change_24h_percent': '24h',
    'trading_volume_24h': '24h_volume',
    'market_cap_usd': 'mkt_cap',
    'timestamp': 'date'
}, inplace=True)

df_sample['date'] = pd.to_datetime(df_sample['date']).dt.date
df_sample['symbol'] = df_sample['coin'].str.lower().str[:3]  # crude fallback
df_sample['1h'] = 0.0  # or np.nan if unavailable
df_sample['7d'] = 0.0  # or np.nan if unavailable
# df_sample['shock_z'] = df_sample['volatility_index'] / 20 + df_sample['news_impact_score'] / 5
# df_sample['regime_shock'] = (df_sample['shock_z'] > 1.5).astype(int)


In [18]:
df_sample.head()


,date,coin,price,24h,24h_volume,mkt_cap,social_sentiment_score,news_sentiment_score,news_impact_score,social_mentions_count,fear_greed_index,volatility_index,rsi_technical_indicator,prediction_confidence,symbol,1h,7d
0,2025-06-04,Algorand,0.3427,-5.35,1716266.10,1.762124e+09,0.367,0.374,1.87,13,53.2,95.1,37.2,78.1,alg,0.0,0.0
1,2025-06-04,Cosmos,12.0420,5.14,10520739.91,2.099180e+11,-0.278,-0.107,1.01,600,43.5,76.7,65.0,66.7,cos,0.0,0.0
2,2025-06-04,Cosmos,11.7675,-6.12,642191.11,1.755370e+11,-0.255,0.211,5.69,279,49.1,60.4,32.3,77.4,cos,0.0,0.0
3,2025-06-04,Ethereum,2861.2829,-11.54,5356227.76,4.786420e+13,-0.531,-0.081,5.11,3504,37.0,100.0,63.0,81.7,eth,0.0,0.0
4,2025-06-04,Solana,95.3583,5.79,735971.56,2.667610e+11,0.369,0.248,1.82,3236,61.7,67.5,55.4,81.8,sol,0.0,0.0


In [19]:
df_sample.info()
df_sample.to_csv("../sample/crypto_sample_test.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2063 entries, 0 to 2062
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     2063 non-null   object 
 1   coin                     2063 non-null   object 
 2   price                    2063 non-null   float64
 3   24h                      2063 non-null   float64
 4   24h_volume               2063 non-null   float64
 5   mkt_cap                  2063 non-null   float64
 6   social_sentiment_score   2063 non-null   float64
 7   news_sentiment_score     2063 non-null   float64
 8   news_impact_score        2063 non-null   float64
 9   social_mentions_count    2063 non-null   int64  
 10  fear_greed_index         2063 non-null   float64
 11  volatility_index         2063 non-null   float64
 12  rsi_technical_indicator  2063 non-null   float64
 13  prediction_confidence    2063 non-null   float64
 14  symbol                  

In [ ]:
import os

def generate_tree(root_dir, prefix="", level=0, max_depth=3):
    if level >= max_depth:
        return ""
    entries = sorted(os.listdir(root_dir))
    tree = ""
    for i, entry in enumerate(entries):
        path = os.path.join(root_dir, entry)
        connector = "└── " if i == len(entries) - 1 else "├── "
        tree += f"{prefix}{connector}{entry}\n"
        if os.path.isdir(path):
            extension = "    " if i == len(entries) - 1 else "│   "
            tree += generate_tree(path, prefix + extension, level + 1, max_depth)
    return tree


root = ".."  # Change this to your project folder name
print("```\n" + root + "/")
print(generate_tree(root) + "```")

```
../
├── Cryto_v2.py
├── EDA
│   ├── Cryto_v2.ipynb
│   └── crypto_eda.ipynb
├── Handoff-Note.md
├── Include
├── Lib
│   └── site-packages
│       ├── .DS_Store
│       ├── Cython
│       │   ├── Build
│       │   │   ├── BuildExecutable.py
│       │   │   ├── Cache.py
│       │   │   ├── Cythonize.py
│       │   │   ├── Dependencies.py
│       │   │   ├── Distutils.py
│       │   │   ├── Inline.py
│       │   │   ├── IpythonMagic.py
│       │   │   ├── SharedModule.py
│       │   │   ├── Tests
│       │   │   │   ├── TestCyCache.py
│       │   │   │   ├── TestCythonizeArgsParser.py
│       │   │   │   ├── TestDependencies.py
│       │   │   │   ├── TestInline.py
│       │   │   │   ├── TestIpythonMagic.py
│       │   │   │   ├── TestRecythonize.py
│       │   │   │   ├── TestStripLiterals.py
│       │   │   │   ├── __init__.py
│       │   │   │   └── __pycache__
│       │   │   │       ├── TestCyCache.cpython-312.pyc
│       │   │   │       ├── TestCythonizeArgsParser.cpython-312.p